# 01 · Canonical data, validation, and provenance

**Goal:** inspect schema v1, understand stable IDs, verify that invalid data is rejected, and see how image metadata joins to a recipe through explicit keys. No images are downloaded.

## Step 1 — Load the bundled JSON fixture

The bundled resource is available even after installing a wheel and starting Jupyter from a different directory.

In [ ]:
import copy
import json
from importlib.resources import files
from savoroute.data.catalog import CatalogError, load_catalog, parse_catalog

text = files("savoroute.resources").joinpath("demo_catalog.json").read_text()
payload = json.loads(text)
print("Envelope:", sorted(payload))
print("First record fields:", sorted(payload["items"][0]))

## Step 2 — Convert JSON into immutable domain objects

In [ ]:
catalog = parse_catalog(payload)
first = catalog.items[0]
print("ID:", first.item_id)
print("Kind:", first.kind)
print("Cuisines:", first.cuisines)
print("Provenance label:", first.source)
assert catalog == load_catalog()

## Step 3 — Reject malformed records

The parser rejects unknown fields and duplicate IDs.

In [ ]:
bad = copy.deepcopy(payload)
bad["items"].append(copy.deepcopy(bad["items"][0]))
try:
    parse_catalog(bad)
except CatalogError as exc:
    print("Expected error:", exc)
else:
    raise AssertionError("Duplicate ID was not rejected")

## Step 4 — Explicit item/media association

This is a *metadata-only* example of a future image reference. No image file is included, fetched, captioned, or embedded. A recipe is associated with an image through `item_id`, never by filename ordering.

In [ ]:
example = copy.deepcopy(payload)
example["media"] = [{
    "media_id": "img001", "item_id": "p001", "kind": "image",
    "uri": "images/example.png", "source": "synthetic_demo"
}]
linked = parse_catalog(example)
for media in linked.media:
    print(media.media_id, "belongs to", linked.get(media.item_id).title)
assert linked.media[0].item_id == "p001"

## Step 5 — Broken foreign keys fail validation

In [ ]:
broken = copy.deepcopy(example)
broken["media"][0]["item_id"] = "p999"
try:
    parse_catalog(broken)
except CatalogError as exc:
    print("Expected error:", exc)
else:
    raise AssertionError("Dangling media foreign key was not rejected")

**Note:** `source` records the dataset label; field-level evidence and media licenses are not modeled in the current format. Next: [`02_lexical_retrieval.ipynb`](02_lexical_retrieval.ipynb).